# Maia Lite 0.3.0 — Auditoria final SFT v2.2

Executar **antes** do LoRA. Nenhuma alteração nos datasets originais. O relatório de aprovação (`audit_pass.json`) só é criado quando as verificações automáticas passam e a revisão humana é explicitamente confirmada. A revisão de licença e de conteúdo não é automatizável integralmente.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
from pathlib import Path
import json, hashlib, re, unicodedata, collections, random, datetime, csv
ROOT=Path("/content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat")
DATA=ROOT/'datasets/sft_v2_2_aya'
TRAIN=DATA/'sft_v2_2_train.jsonl'; VAL=DATA/'sft_v2_2_validation.jsonl'
REPORT=DATA/'audit_v2_2';REPORT.mkdir(exist_ok=True)
for p in (TRAIN,VAL): assert p.is_file(),f'Arquivo ausente: {p}'
def read(p):
    with p.open(encoding='utf8') as f:
        for line,n in ((line,n) for n,line in enumerate(f,1)):
            if line.strip():
                try: yield json.loads(line)
                except Exception as e: raise ValueError(f'{p}:{n}: {e}')
def sha(p):
    h=hashlib.sha256()
    with p.open('rb') as f:
        for chunk in iter(lambda:f.read(1<<20),b''):h.update(chunk)
    return h.hexdigest()
def norm(s):return ' '.join(unicodedata.normalize('NFKC',s).casefold().split())
train=list(read(TRAIN));val=list(read(VAL))
print('Treino:',len(train),'Validação:',len(val))
print('Hashes:',sha(TRAIN),sha(VAL))

## Verificações estruturais e de vazamento

A duplicação exata é verificada por prompt normalizado e por par prompt/resposta. O `groupId` e `paperId` são comparados quando presentes. Sobreposição semântica aproximada não é eliminada automaticamente.

In [ ]:
issues=[]
def flag(kind,detail):issues.append({'kind':kind,'detail':str(detail)[:400]})
for split,rows in [('train',train),('validation',val)]:
    ids=set();prompts=set();pairs=set()
    for i,r in enumerate(rows):
        for k in ('id','prompt','completion','language'):
            if k not in r or not isinstance(r[k],str) or not r[k].strip():flag('missing_field',f'{split}:{i}:{k}')
        if any(k not in r for k in ('id','prompt','completion','language')):continue
        if r['language'] not in ('pt','en','es'):flag('invalid_language',f'{split}:{i}:{r["language"]}')
        if r['id'] in ids:flag('duplicate_id',f'{split}:{r["id"]}')
        ids.add(r['id'])
        p=norm(r['prompt']);pair=(p,norm(r['completion']))
        if p in prompts:flag('duplicate_prompt',f'{split}:{i}')
        if pair in pairs:flag('duplicate_pair',f'{split}:{i}')
        prompts.add(p);pairs.add(pair)
        if p==norm(r['completion']):flag('echo_answer',f'{split}:{i}')
for field in ('id','groupId','paperId'):
    x={str(r[field]) for r in train if r.get(field) not in (None,'')}
    y={str(r[field]) for r in val if r.get(field) not in (None,'')}
    common=x&y
    if common:flag('split_overlap_'+field,f'{len(common)} overlaps; sample={list(sorted(common))[:5]}')
ptrain={norm(r['prompt']) for r in train if isinstance(r.get('prompt'),str)}
pval={norm(r['prompt']) for r in val if isinstance(r.get('prompt'),str)}
if ptrain&pval:flag('prompt_leak',f'{len(ptrain&pval)} overlaps')
counts=collections.Counter(r.get('language') for r in train)
source_counts=collections.Counter(str(r.get('source','unknown')) for r in train)
print('Idiomas:',dict(counts),'Fontes:',dict(source_counts))
print('Problemas automáticos:',len(issues));print(issues[:12])

## Amostragem estratificada para revisão humana

Revise perguntas, respostas, idioma declarado, integridade de dados científicos e eventuais condições de licenciamento. **Não marque aprovação sem revisar.**

In [ ]:
random.seed(20261118)
buckets=collections.defaultdict(list)
for r in train:buckets[(r.get('language'),str(r.get('source','unknown')))].append(r)
sample=[]
for bucket,rows in sorted(buckets.items(),key=lambda x:str(x[0])):
    sample.extend(random.sample(rows,min(20,len(rows))))
sample_file=REPORT/'review_sample.jsonl'
with sample_file.open('w',encoding='utf8') as f:
    for r in sample:f.write(json.dumps(r,ensure_ascii=False)+'\n')
print('Amostras para revisão:',len(sample),'em',sample_file)
print('Amostra inicial:',json.dumps(sample[0],ensure_ascii=False)[:1200] if sample else '')

## Parecer e congelamento

Preencha `HUMAN_REVIEW_APPROVED=True` **somente após** examinar a amostra, confirmar a procedência e as licenças de cada fonte, e aceitar os riscos residuais. O treinamento verificará hashes e aprovação. O estado `INCOMPLETE` do manifesto anterior reflete apenas cinco exemplos faltantes, não bloqueia esta auditoria.

In [ ]:
HUMAN_REVIEW_APPROVED=False  # alterar manualmente após revisão
LICENSE_REVIEW_APPROVED=False  # confirmar termos de cada fonte e de cada paper
summary={'created_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),
         'train_count':len(train),'validation_count':len(val),'train_by_language':dict(counts),
         'train_by_source':dict(source_counts),'issues_count':len(issues),
         'train_sha256':sha(TRAIN),'validation_sha256':sha(VAL),
         'human_review_approved':HUMAN_REVIEW_APPROVED,
         'license_review_approved':LICENSE_REVIEW_APPROVED}
(REPORT/'audit_report.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2))
with (REPORT/'audit_issues.jsonl').open('w',encoding='utf8') as f:
    for x in issues:f.write(json.dumps(x,ensure_ascii=False)+'\n')
PASS=REPORT/'audit_pass.json'
if not issues and HUMAN_REVIEW_APPROVED and LICENSE_REVIEW_APPROVED:
    PASS.write_text(json.dumps(summary,ensure_ascii=False,indent=2))
    print('AUDITORIA APROVADA:',PASS)
else:
    if PASS.exists():PASS.unlink() # não reutilizar aprovação obsoleta
    print('PENDENTE:',len(issues),'problemas; revisão humana:',HUMAN_REVIEW_APPROVED,'licenças:',LICENSE_REVIEW_APPROVED)
print('Relatório:',REPORT/'audit_report.json')